Install required libraries

In [2]:
!pip install faiss-cpu sentence-transformers transformers PyPDF2

In [3]:
import os
from pathlib import Path

HOME = Path.home()

MODELS_DIR = HOME / "ai_models"
UPLOADS_DIR = HOME / "ai_uploads"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

## Upload files to colab

In **JupyterLab**, 

1. Select **File Browser** on left.
2. Browse to the **UPLOAD_DIR** directory
3. Upload several PDF files

Suggested files:
 - NIST CSF - https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf
 - NIST 800-53 - https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-53r5.pdf
 - OWASP Top 10 - blob:https://github.com/455a0203-e7d6-47dd-b427-37da96ef0718
 - Cisco IOS XE configuration guides
 - OSPF configuration guides - https://www.cisco.com/c/en/us/td/docs/ios-xml/ios/iproute_ospf/configuration/xe-16/iro-xe-16-book.pdf
 - BGP configuration guides
 - VLAN configuration guides
 - NAT configuration guides
 - Azure Virtual Networks
 - Azure VPN Gateway
 - Azure Load Balancer
 - Azure Availability Zones
 - systemd docs
 - ssh docs - https://docs.ssh.com/manuals/client-user/71/Client-user.pdf
 - firewalld docs
 - SELinux documentation
 - RAG paper - https://arxiv.org/pdf/2312.10997
 - Transformer paper
 - Vector database article - https://arxiv.org/pdf/2310.11703
 - Embeddings article

## Load the Base LLM

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Slightly larger, better quality (needs GPU ideally)
model_name = "google/flan-t5-base"

# Even smaller (if students are on CPU only and struggling)
# model_name = "google/flan-t5-small"  # keep this as default

# For extra  experiments, there is also
# model_name = "facebook/bart-base"  # medium size; good if GPU present


tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir = MODELS_DIR
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    cache_dir = MODELS_DIR
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Prepare questions and ask - Before RAG

In [5]:
questions = [
    "What are the main components of the Solar System?",
    "How does the Internet work at a high level?",
    "What are some applications of artificial intelligence?",
    "What are the 2027 FSA annual maximum?"
]

In [6]:
questions = [
    "What is a planet?",
    "What is a natural satellite?",
    "Which planets have ring systems?",
    "What objects are included in the Solar System?",
    "How are planets different from small Solar System bodies?",
    "Why are natural satellites important within the Solar System?",
    "What types of objects orbit the Sun?",
    "What are the major categories of objects found in the Solar System?",
    "Besides planets, what other celestial bodies exist in the Solar System?",
    "Compare planets, moons, and small Solar System bodies.",
    "Describe the structure of the Solar System and the objects within it."
]

In [7]:
questions = [
    "How do I enable OSPF on an interface?",
    "What is the difference between OSPF area 0 and a stub area?",
    "What Cisco commands verify BGP neighbors?",
    "What is the principle of least privilege?",
    "Which CIS control addresses vulnerability management?",
    "What are the major OWASP web application risks?",
    "What is the difference between a Load Balancer and an Application Gateway?",
    "How does zone redundancy work?",
    "What are the components of a virtual network?",
    "How do I restart a service?",
    "How does SELinux differ from standard Unix permissions?",
    "What is the purpose of systemd?",
    "What are embeddings?",
    "How does retrieval augmentation work?",
    "What is the purpose of chunking?",
    "Why does overlap improve retrieval?"
]

In [8]:
def ask(model, question):
    inputs = tokenizer(question, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=300, num_beams=4)
    print("Q:", question)
    print("A:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("-" * 80)

In [9]:
for q in questions:
    ask(model, q)

Q: How do I enable OSPF on an interface?
A: To enable OSPF on an interface, you first need to enable OSPF on the interface.
--------------------------------------------------------------------------------
Q: What is the difference between OSPF area 0 and a stub area?
A: OSPF area 0 is a stub area
--------------------------------------------------------------------------------
Q: What Cisco commands verify BGP neighbors?
A: bgp
--------------------------------------------------------------------------------
Q: What is the principle of least privilege?
A: rule of law
--------------------------------------------------------------------------------
Q: Which CIS control addresses vulnerability management?
A: cisco
--------------------------------------------------------------------------------
Q: What are the major OWASP web application risks?
A: OWASP is a software company that develops and maintains web applications. OWASP is a software company that develops and maintains web applications

## Extract text from PDF

In [10]:
import os
import PyPDF2

documents = []

for filename in os.listdir(UPLOAD_DIR):
    if not filename.lower().endswith(".pdf"):
        continue

    filepath = os.path.join(UPLOAD_DIR, filename)

    with open(filepath, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        documents.append((filename, text))

print("Documents loaded:")
for filename, _ in documents:
    print(filename)

Documents loaded:
NIST.SP.800-53r5.pdf
OWASP Top Ten 2025.pdf
2310.11703v4.pdf
Client-user.pdf
2312.10997v5.pdf
AzureNetworkSecurity_v3_Feb2015.pdf
NIST.CSWP.29.pdf


## Build a Vector Store from uploaded documents

In [11]:
chunks = []

# chunk on newline
# for (filename, text) in documents:
#     for paragraph in text.split("\n"):
#         paragraph = paragraph.strip()
#         if len(paragraph) > 120:  # ignore tiny lines
#             chunks.append(paragraph)

# chunk on 1000 bytes - tune or or down as needed
chunk_size = 500
overlap = 200
for filename, text in documents:
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]

        if len(chunk.strip()) > 100:
            chunks.append(chunk)

| Chunk Size | Usage |
| -- | -- |
| 100-300 chars | Usually too small |
| 500-1000 chars | Common for simple RAG labs |
| 1000-2000 chars | Very common production setting |
| 2000-4000 chars | Larger context retrieval |
| 4000+ chars | Often becomes inefficient |



Choose the embedding model (Only choose one)

In [12]:
from sentence_transformers import SentenceTransformer

# Baseline (what you’re using now)
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    cache_folder = MODELS_DIR
)

# Alternative 1: slightly better quality, still small
# embedder = SentenceTransformer(
#     "sentence-transformers/all-mpnet-base-v2",
#     cache_folder = MODELS_DIR
# ))

# Alternative 2: e5-style embeddings (good for RAG)
# embedder = SentenceTransformer(
#     "intfloat/e5-small-v2",
#     cache_folder = MODELS_DIR
# ))

# Alternative 3: BGE small (popular in RAG examples)
# embedder = SentenceTransformer(
#     "BAAI/bge-small-en-v1.5",
#     cache_folder = MODELS_DIR
# ))

# Alternative 4: mpnet (larger and slower)
# embedder = SentenceTransformer(
#     "sentence-transformers/all-mpnet-base-v2",
#     cache_folder = MODELS_DIR
# )

chunk_embeddings = embedder.encode(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))

Setup Retrieval Function

In [14]:
# how many chunks to evaluate
# adjust k as needed from 3-10. 5-8 suggested

def retrieve(question, k=3):
    q_embed = embedder.encode([question])
    distances, indices = index.search(np.array(q_embed), k)
    return [chunks[i] for i in indices[0]]

Combined RAG Answer Generator

In [15]:
def ask_rag(model, question, k=3):
    retrieved = retrieve(question, k)
    context = "\n".join(retrieved)

    prompt = (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=300, num_beams=4)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("Q:", question)
    print("A:", answer)
    print("-" * 80)

Ask same questions - with RAG

In [16]:
for q in questions:
    ask_rag(model, q)

Q: How do I enable OSPF on an interface?
A: specify ::
--------------------------------------------------------------------------------
Q: What is the difference between OSPF area 0 and a stub area?
A: Out- of-band channels do not have the same vulnerability or exposure as in -band channels
--------------------------------------------------------------------------------
Q: What Cisco commands verify BGP neighbors?
A: c Exchange: BGP Security and DDoS Mitigation. (Nati onal Institute of Standards and Technology, Gaithersburg, MD), NIST Special Publication (SP) 800 -189. https://doi.org/10.6028/NIST.SP.800- 189 [SP 800 192] Yaga DJ, Kuhn R, Hu VC (2017) Verification and Test Methods for Access Control Policies/Models . (National Institute of Standards and Technology, Gaithersburg, MD), NIST Special Publication (SP) 800 -189. https://doi.org/10.6028/NIST.SP.800- 189 [SP 800 192] Cooper DA, MacGregor lient-g3 command-line command keywords are listed below. Shorthands longer than three lett

# Cleanup files when finished

Only clean up temporary files when finished. Models and files will need to be redownloaded if used in the future.

In [1]:
# Cleanup
# import shutil

# shutil.rmtree(MODELS_DIR)
# shutil.rmtree(UPLOAD_DIR)

# print("Temporary files removed")

NameError: name 'MODELS_DIR' is not defined

Ask both models simultaneously

In [17]:
import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Text(
    placeholder="Ask a question...",
    description="Question:",
    layout=widgets.Layout(width='80%')
)

button = widgets.Button(
    description="Ask"
)

output = widgets.Output()

def submit_question(_):

    question = question_box.value.strip()

    if not question:
        return

    with output:
        output.clear_output()

        print("=" * 70)
        print("QUESTION")
        print("=" * 70)
        print(question)
        print()

        # Uncomment to see RAG content
        # retrieved = retrieve(question, k=5)
        # print("\nRETRIEVED CONTEXT")
        # print("-"*50)
    
        # for i, chunk in enumerate(retrieved, 1):
        #     print(f"[Chunk {i}]")
        #     print(chunk[:300])
        #     print()

        print("=" * 70)
        print("LLM ONLY")
        print("=" * 70)

        try:
            print(ask(model, question))
        except Exception as e:
            print("Error:", e)

        print()
        print("=" * 70)
        print("RAG")
        print("=" * 70)

        try:
            print(ask_rag(model, question))
        except Exception as e:
            print("Error:", e)

button.on_click(submit_question)

display(question_box)
display(button)
display(output)

Text(value='', description='Question:', layout=Layout(width='80%'), placeholder='Ask a question...')

Button(description='Ask', style=ButtonStyle())

Output()